In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("MY_GITHUB_TOKEN")
os.environ["GIT_TERMINAL_PROMPT"] = "0"          # fail fast if auth fails, never prompt

# 2) Fresh clone into the Kaggle working dir (NOT /content — that's Colab)
!rm -rf sports_prediction_model
!git clone -q https://github.com/andrewkemmer/sports_prediction_model.git sports_prediction_model

# 3) Install every dependency the pipeline imports (backend, not the Streamlit frontend)
!pip install -q nflreadpy gitpython polars scikit-learn \
    lightgbm xgboost pandas numpy joblib requests
print("setup done")



import os

# --- OPTIONAL overrides (omit everything for a normal run) ---

# Slate target season: the season the board predicts and the sealed-2025 gate
# holds out. Omit = current calendar year; set only for a backfill.
os.environ["NFL_START_SEASON"] = "2019"   # <- your start selector
os.environ["NFL_END_SEASON"]   = "2025"   # <- your end selector
os.environ["NFL_SLATE_SEASON"] = "2026"   # the board's target season

# NOTE: unlike MLB there is no start/end window or full-repull knob — NFL Phase 1
# always rebuilds the decided frame from the full 2019-2025 history.
# (Dry-run without pushing: add "--no-push" to the command in the next cell.)

print("run options set")




import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"
cmd = ["python", "nfl-backend/backend/master_pipeline.py"]

result = subprocess.run(cmd, cwd=repo, env=os.environ.copy(), capture_output=False)

# The NFL pipeline writes artifacts locally. Delivery is explicit here so a
# successful model run cannot be reported as delivered when GitHub is stale.
if result.returncode != 0:
    raise SystemExit(f"Pipeline failed with exit code {result.returncode}")

from datetime import datetime, timezone
from urllib.parse import quote

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
repo_url = "https://github.com/andrewkemmer/sports_prediction_model.git"
token = os.environ.get("GITHUB_TOKEN", "").strip()
if not token:
    raise SystemExit("GITHUB_TOKEN is required for NFL artifact delivery")

# Authenticate only the clone's remote; never print the token.
auth_url = f"https://x-access-token:{quote(token, safe='')}@github.com/andrewkemmer/sports_prediction_model.git"
subprocess.run(["git", "remote", "set-url", "origin", auth_url], cwd=repo, check=True)

# Only these dated NFL serving families and the production model bundle may
# be delivered. No MLB path, source code, experiment, or OOF store can enter
# this commit.
production_names = [
    f"nfl_moneyline_v1_{run_stamp}.json",
    f"nfl_calibration_{run_stamp}.json",
    f"nfl_predictions_history_{run_stamp}.csv",
    f"nfl_power_rankings_{run_stamp}.csv",
    f"nfl_run_engine_markets_{run_stamp}.csv",
    f"nfl_run_engine_markets_{run_stamp}.meta.json",
    f"nfl_run_engine_monitor_{run_stamp}.json",
    f"nfl_qb_matchup_{run_stamp}.json",
    f"nfl_feature_v1_{run_stamp}.json",
    f"nfl_model_monitor_{run_stamp}.json",
]
production_paths = [
    f"nfl-backend/data_delivery/{name}" for name in production_names
] + ["nfl-backend/data_delivery/models/nfl_ensemble_latest.joblib"]

for rel in production_paths:
    if not os.path.isfile(os.path.join(repo, rel)):
        raise SystemExit(f"NFL delivery artifact missing: {rel}")

# Rebase-and-push is bounded and race-safe. Every retry stages only the
# allow-listed NFL paths, then verifies the same paths on origin/main.
last_error = None
for attempt in range(1, 4):
    try:
        subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
        subprocess.run(["git", "rebase", "origin/main"], cwd=repo, check=True)
        subprocess.run(["git", "reset"], cwd=repo, check=True)
        subprocess.run(["git", "add", "-f", "--", *production_paths], cwd=repo, check=True)
        staged = subprocess.run(
            ["git", "diff", "--cached", "--name-only"],
            cwd=repo, check=True, capture_output=True, text=True,
        ).stdout.splitlines()
        if not staged or any(not p.startswith("nfl-backend/data_delivery/") for p in staged):
            raise SystemExit(f"NFL delivery scope violation: {staged}")
        if staged:
            subprocess.run(["git", "commit", "-m", f"NFL production artifacts: {run_stamp}"],
                           cwd=repo, check=True)
        subprocess.run(["git", "push", "origin", "main"], cwd=repo, check=True)
        subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
        remote_paths = set(subprocess.run(
            ["git", "ls-tree", "-r", "--name-only", "origin/main", "nfl-backend/data_delivery"],
            cwd=repo, check=True, capture_output=True, text=True,
        ).stdout.splitlines())
        missing = [p for p in production_paths if p not in remote_paths]
        if missing:
            raise RuntimeError(f"remote NFL artifact verification failed: {missing}")
        print(f"NFL artifacts delivered and remotely verified: {run_stamp} ({len(staged)} files)")
        break
    except Exception as exc:
        last_error = exc
        if attempt == 3:
            raise SystemExit(f"NFL artifact delivery failed after 3 attempts: {exc}")
        print(f"NFL delivery attempt {attempt} failed; retrying against origin/main")
else:
    raise SystemExit(f"NFL artifact delivery failed: {last_error}")



import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"

# Confirm main moved: fetch and compare HEAD to the pre-run HEAD
check = subprocess.run(
    ["git", "log", "--oneline", "-1"],
    cwd=repo, capture_output=True, text=True,
)
print("Repo HEAD after run:", check.stdout.strip())

# Sanity: newest dated artifact on main
ls = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "origin/main"],
    cwd=repo, capture_output=True, text=True,
)
datelated = [f for f in ls.stdout.splitlines() if "nfl-backend/data_delivery/" in f and "_202" in f]
print("Latest dated artifacts:", sorted(datelated)[-3:] if datelated else "none found")
